In [29]:
import emat
import pandas as pd
import pickle

emat.require_version('0.5.1')

emat 0.6.0, plotly 6.1.2


### Remote I/O

In [30]:
pickle_filename = './data/processed/core_results_version_02/meta_model_pag.pkl'
output_filename = "./data/processed/meta-model-regression-estimation-results.csv"

### Data Reads

In [31]:
with open(pickle_filename, 'rb') as file:
    metamodel = pickle.load(file)

### Make Dataframes

In [32]:
regression_df = metamodel.function.regression.lr.coefficients_summary()
regression_df.reset_index(drop = False, inplace = True)
regression_df = regression_df.rename(columns={'level_0': 'model_name', 'level_1': 'variable_name'})
regression_df.head()

,model_name,variable_name,Coefficient,StdError,t-Statistic,p
0,AM Travel Time Index,AV_Rates,-0.242535,0.010686,-22.695643,0.000000
1,AM Travel Time Index,EV_Rates,0.034419,0.011101,3.100604,0.002480
2,AM Travel Time Index,Ecommerce,0.154151,0.054144,2.847087,0.005308
3,AM Travel Time Index,HH_EMP_Distribution,0.007704,0.010717,0.718902,0.473798
4,AM Travel Time Index,HH_EMP_Growth,0.107444,0.010876,9.878630,0.000000


In [33]:
r_squared_df = pd.DataFrame(metamodel.function.regression.estimators_[0].r2, columns = ["r-squared"])
r_squared_df = r_squared_df.reset_index(drop = False).rename(columns={'index': 'model_name'})
r_squared_df

,model_name,r-squared
0,Auto Mode Share,0.973897
1,Bike Mode Share,0.962534
2,Transit Mode Share,0.963846
3,Walk Mode Share,0.902791
4,Transit PMT,0.917603
5,Regional VMT,0.669360
6,Regional VMT per Capita,0.565693
7,Regional VHT,0.703464
8,Regional VHT per Capita,0.514490
9,Average Commute Time by Auto,0.743420


In [34]:
output_df = pd.merge(
    left = regression_df,
    right = r_squared_df,
    how = 'left',
    on = 'model_name'
)
output_df["model_id"] = pd.factorize(output_df["model_name"])[0] + 1
output_df = output_df[["model_id", "model_name", "r-squared", "variable_name", "Coefficient", "StdError", "t-Statistic", "p"]]
output_df.head()

,model_id,model_name,r-squared,variable_name,Coefficient,StdError,t-Statistic,p
0,1,AM Travel Time Index,0.876879,AV_Rates,-0.242535,0.010686,-22.695643,0.000000
1,1,AM Travel Time Index,0.876879,EV_Rates,0.034419,0.011101,3.100604,0.002480
2,1,AM Travel Time Index,0.876879,Ecommerce,0.154151,0.054144,2.847087,0.005308
3,1,AM Travel Time Index,0.876879,HH_EMP_Distribution,0.007704,0.010717,0.718902,0.473798
4,1,AM Travel Time Index,0.876879,HH_EMP_Growth,0.107444,0.010876,9.878630,0.000000


In [35]:
output_df.to_csv(output_filename, index=False)